In [ ]:
'''
To run this shole jupyter notebook as a python script follow this:

(1) activate conda environemt
(2) go to where this notebook is located in your computer
(3) use `python` to enter python with in your shell/terminal
(4) follow the above syntax


from json import load

filename = 'step_2_prep_maps_and_netmats.ipynb'
with open(filename) as fp:
    nb = load(fp)

for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        source = ''.join(line for line in cell['source'] if not line.startswith('%'))
        exec(source, globals(), locals())
'''

In [4]:
import pandas as pd
import nibabel as nb
import numpy as np
import yaml
import os
import argparse
import sys
# sys.path.append('../')
# sys.path.append('./')
# sys.path.append('../../')

if "/Users/snaranjo" in os.getcwd():
    local_pc_flag = "/Users/snaranjo/Desktop/neurotranslate/mount_point"
else:
    local_pc_flag=""

ico = 6 # 6
sub_ico = 2 # 2
split = 'train' #validation # or test
translation = 'ICAd15_schfd100'
dataset = 'ABCD_v6'
# nm_fs_data = config['data']['fs_data_path'] # d15_fs_LR
sub_ids_path = f'{local_pc_flag}/ceph/chpc/shared/janine_bijsterbosch_group/WAPIAW_2026/qc/5min_pconn_subjects.txt' #f'{local_pc_flag}/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD_v6/schaefer_mats/netmat_d100'
parcellation_type = 'full'
parcellation_name = 'schaefer'
num_vertices = 153 #config['sub_ico_{}'.format(sub_ico)]['num_vertices'] # sub_ico_2
num_patches = 320 #config['sub_ico_{}'.format(sub_ico)]['num_patches'] # sub_ico_2
chosen_hemi = '1L' #config['data']['hemisphere'] #1L or 1R or 2 for both

path_to_data = f'{local_pc_flag}/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD_v6/ICA_maps/ICA_d20_ico06' #ico res
skipped_subject_path=f'{local_pc_flag}//ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/surf2netmat/utils/subj_ids/'
output_folder = f'{local_pc_flag}/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD_v6/ICA_maps/ICA_d20_ico02' #whatever subico you have
# output_folder_netmat = f'{local_pc_flag}/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD_v6/schaefer_mats/netmat_d100' #

In [2]:
get_subject_list = pd.read_csv(f"{local_pc_flag}/ceph/chpc/shared/janine_bijsterbosch_group/WAPIAW_2026/qc/5min_pconn_subjects.txt", sep=' ', header=None)[0].to_numpy()#.to_list() #first and only column
print(f"Number of subjects: {len(get_subject_list)}, {get_subject_list[-5:-1]}")

fam_id_files=pd.read_csv(f"{local_pc_flag}/ceph/chpc/rcif_datasets/abcd/ABCD-6.0/phenotypes/ab/ab_g_stc.tsv", sep='\t')
get_relevant_columns = fam_id_files[["participant_id", "ab_g_stc__design_id__fam__gen","ab_g_stc__design_id__birth__gen", "ab_g_stc__design_id__group"]]

print(get_relevant_columns.shape)
mask_to_match_original_subIDlist = np.isin(get_relevant_columns["participant_id"], get_subject_list)
print(mask_to_match_original_subIDlist.shape, mask_to_match_original_subIDlist.sum())

get_only_subjlist_match_subjs=get_relevant_columns[mask_to_match_original_subIDlist]
get_only_subjlist_match_subjs.sort_values(by=["ab_g_stc__design_id__fam__gen"]).dropna() #order by family id to match singletons and twins/triplets+
assert get_only_subjlist_match_subjs.shape[0] == len(get_subject_list)
# print(get_only_subjlist_match_subjs.head())

counts = get_only_subjlist_match_subjs["ab_g_stc__design_id__fam__gen"].value_counts() #each value and its frequency
single_counts = counts[counts == 1].index #index of where this is true
two_counts = counts[counts == 2].index
three_counts = counts[counts == 3].index
more_counts = counts[counts > 3].index
print(f"Only once:{len(single_counts)}, Twice:{len(two_counts)}, Thrice:{len(three_counts)}, More:{len(more_counts)}")
singletons = get_only_subjlist_match_subjs[get_only_subjlist_match_subjs["ab_g_stc__design_id__fam__gen"].isin(single_counts)]
print(three_counts)
# print(get_only_subjlist_match_subjs["ab_g_stc__design_id__fam__gen"])
twins = get_only_subjlist_match_subjs[get_only_subjlist_match_subjs["ab_g_stc__design_id__fam__gen"].isin(two_counts)]
triplets = get_only_subjlist_match_subjs[get_only_subjlist_match_subjs["ab_g_stc__design_id__fam__gen"].isin(three_counts)]

#now make train/val/test split respecting this
subj_list_singles=singletons["participant_id"].to_list()
subj_list_twins=twins["participant_id"].to_list()
subj_list_triplets=triplets["participant_id"].to_list()

#order them and save
subject_list_subset=subj_list_twins
sibling_type="twins"

from_main_df_get_subsets_mask = np.isin(get_only_subjlist_match_subjs["participant_id"], subject_list_subset)
order_them = get_only_subjlist_match_subjs[from_main_df_get_subsets_mask]
order_them=order_them.sort_values(by="ab_g_stc__design_id__fam__gen")
directory=f"{local_pc_flag}/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/surf2netmat/utils/subj_ids"

flag_seperate_by_famID=False
if flag_seperate_by_famID:
    order_them.to_csv(f"{directory}/ABCDv6_{sibling_type}.csv")
# print(order_them)

Number of subjects: 3234, ['sub-YR1LYX8E' 'sub-JJMH6ZDT' 'sub-T2D0E2LE' 'sub-GB1LBZ5W']
(11868, 4)
(11868,) 3234
Only once:2631, Twice:276, Thrice:5, More:0
Float64Index([629.0, 7443.0, 4866.0, 1080.0, 3304.0], dtype='float64')


In [3]:
def fcn_generate_subject_split(single_df: pd.DataFrame, twins_df: pd.DataFrame, triplets_df: pd.DataFrame, train_split_portion:float=0.8, random_seed: int=123):
    assert train_split_portion < 1 and train_split_portion > 0, "Train split must be 0<x<1."

    '''Goal here is to have a train/val/test split where subject relationship is respected.
    If a twin/triplet is in a split, the other siblings must also be in that split.'''
    total_N = single_df.shape[0] + twins_df.shape[0] + triplets_df.shape[0]
    import math
    train_N= math.floor(total_N * train_split_portion) #if its even we're good
    # added_part_cond = 1 if train_N % 2 != 0 else 0
    left_over = total_N-train_N
    validation_N = left_over // 2
    test_N = left_over - validation_N
    print(f"Based on train_split_proportion ({train_split_portion}): \nTrain:{train_N} \nValidation:{validation_N} \nTest:{test_N}")
    #ensure that these are ordered by family ID where same ID means TWINS/TRIPLETS
    name_of_famID_column="ab_g_stc__design_id__fam__gen"
    name_of_subID_column="participant_id"
    single_df=single_df.sort_values(by=[name_of_famID_column]).reset_index(drop=True)
    twins_df=twins_df.sort_values(by=[name_of_famID_column]).reset_index(drop=True)
    triplets_df=triplets_df.sort_values(by=[name_of_famID_column]).reset_index(drop=True)
    # now that sorted we can split and use that they are ordered for our purposes
    family_to_subjects: dict[str, list] = {} #init dictionary
    for df in [single_df, twins_df, triplets_df]: 
        for fam_id, group in df.groupby(name_of_famID_column): #organize by group 
            subjects = group[name_of_subID_column].to_list() #group size depends on fam, so singletons==1,twins==2,triplets==3
            if fam_id in family_to_subjects:
                family_to_subjects[fam_id].extend(subjects)
            else:
                family_to_subjects[fam_id] = subjects #every fam_id is a key and corresponding list of subjects for that key
    
    #shuffle family
    family_ids = list(family_to_subjects.keys())
    pre_family_ids=family_ids
    from random import shuffle
    # rng = random.Random(random_seed)
    # print(f"Pre-Shuffle: {pre_family_ids}")
    shuffle(family_ids)
    post_family_ids=family_ids
    # print(f"Post-Shuffle: {post_family_ids}")
    assert pre_family_ids != post_family_ids, "Shuffle did not work for some reason. Check this."

    # assign families to splits
    train_ids, validation_ids, test_ids = [],[],[]
    train_count, validation_count, test_count = 0,0,0

    for fam_id in family_ids:
        subjects = family_to_subjects[fam_id] #list of subjects for that famID key
        family_size = len(subjects)

        #starting with train, fill the train/validation/test splits as needed
        if train_count < train_N:
            train_ids.extend(subjects) #assign that list of subjects to train
            train_count += family_size
        elif validation_count < validation_N:
            validation_ids.extend(subjects)
            validation_count += family_size
        else:
            test_ids.extend(subjects)
            test_count += family_size
    
    #report and make sure no overlap in assignment
    print(
        f"Subject counts after family-aware assignment:\n"
        f"Train:{len(train_ids)} \nValidation:{len(validation_ids)} \Test:{len(test_ids)}\n"
        f"Total:{len(train_ids) + len(validation_ids) + len(test_ids)}"
    )
    all_assigned   = train_ids + validation_ids + test_ids
    all_subjects   = (
        single_df[name_of_subID_column].tolist()
        + twins_df[name_of_subID_column].tolist()
        + triplets_df[name_of_subID_column].tolist()
    )
    assert set(all_assigned) == set(all_subjects), \
        "Mismatch: some subjects were lost or duplicated during splitting!"
    assert len(set(train_ids) & set(validation_ids)) == 0,   "Overlap between train and val!"
    assert len(set(train_ids) & set(test_ids)) == 0,  "Overlap between train and test!"
    assert len(set(validation_ids)   & set(test_ids)) == 0,  "Overlap between val and test!"

    return train_ids, validation_ids, test_ids

generate_new_subject_list=False
if generate_new_subject_list:
    single_df = pd.read_csv(f"{directory}/ABCDv6_singles.csv")
    twins_df = pd.read_csv(f"{directory}/ABCDv6_twins.csv")
    triplets_df = pd.read_csv(f"{directory}/ABCDv6_triplets.csv")
    total_N = single_df.shape[0] + twins_df.shape[0] + triplets_df.shape[0]
    # print(single_df.shape, twins_df.shape, triplets_df.shape,total_N)

    train_ids, validation_ids, test_ids = fcn_generate_subject_split(single_df, twins_df, triplets_df)
    #save generated subject lists for train/validation/test split. Then, begin preprocessing of those participants
    ABCDv6_split_ids = pd.DataFrame({
        "participant_ids": np.concatenate((train_ids,validation_ids,test_ids)),
        "assignment": ["train"]*len(train_ids) + ["validation"]*len(validation_ids) + ["test"]*len(test_ids)
    })
    from datetime import date
    todays_date = date.today()
    ABCDv6_split_ids.to_csv(f"{directory}/ABCDv6_split_ids_{todays_date}.csv")

In [ ]:
# Get meshes and paired netmats
ids = pd.read_csv(f"{sub_ids_path}", sep=' ', header=None)
# print(ids) #for each of these subjects, find their mesh if they have one and preprocess then get their netmats

#making sure everyone has a mesh is most important so lets do that then pair them with ther respective netmats
# labels
num_subjects = len(ids)
print('Num of subjects:'+str(num_subjects))

data = [] # list of numpy arrays each is a numpy array version of the shape.gii info
nan_surf_subject_count = 0
sub_ids_nan_vals = []
print(f'Dataset is {dataset}')
if chosen_hemi == '1L':
    print('Left hemisphere was chosen.')
    for i, id in enumerate(ids): # reads in actual id num with 'id' inside the pandas column from the read csv, see above ids variable

        filename = f'{path_to_data}/resamp_{id}.L.shape.gii'
        check_exists = os.path.exists(filename)
        subject_list_skipped=[]
        if check_exists is False:
            print(f'This path did not exist, skipping:{id}')
            subject_list_skipped.append(id)
            continue

        # data is then an array where each element is a shape.gii image
        # agg based on channels, bc if you look below at print, data is [15,49k] so resamp gray ordinates 
        # for each ICA!! I think this then aggregates based on channels? See below if changes
        # from nibabel website, agg_data() is "Aggregate GIFTI data arrays into an ndarray or tuple of ndarray", assuming that makes conversion to numpy easy
        data.append(np.array(nb.load(filename).agg_data()))

        if np.isnan(data[i]).sum() != 0: # so if there are nans
            data_check_nans = np.isnan(data[i]).sum()
            print(f"DATA SUBJ ID:{id}/ iter:{i} NaN Count - {data_check_nans}")

            
            sub_ids_nan_vals.append(id)
            

        if i%300==0:
            # check_load = nb.load(filename)
            # check_agg_data = nb.load(filename).agg_data()
            print('\nLoading GIFTI for subject: {}'.format(i))
            # print('\nChecking data metrics. \n MRI data has loaded number of data arrays: {} \n After Using agg_data(), looking into first tuple: {}'.format(len(check_load.darrays),len(check_agg_data[0])))
            print('\nActual stored data value, presumably aggregated through channels=inputdim:{}'.format(len(data[i])))
        # from sanity checks, I see now that our data values are dim C x TS, our in our case inputdim x TS for each vertex in the sphere
        
        # save list of subject skipped in case needed for the future
        subjects_skipped_nomesh = pd.DataFrame({
                "subIDs": subject_list_skipped
            })
        subjects_skipped_nomesh.to_csv(f"{skipped_subject_path}/subjects_skipped_nomesh.csv")
elif chosen_hemi == '1R':
    print('Right hemisphere was chosen.')
    for i, id in enumerate(ids):
    
        if dataset == "ABCD":
            filename = os.path.join(f'{path_to_data}/resamp_{id}.L.shape.gii')
        elif dataset == "HCPYA_ABCDdr":
            filename = os.path.join(f'{path_to_data}/resamp_{id}.L.shape.gii')
        else:
            filename = os.path.join(path_to_data,'ICA_fs_LR_32',nm_fs_data,'resamp_{}.L.shape.gii'.format(id)) # shape.gii is a metric file for values at each vertex, in our case the ICA values
        
        # check if real, ow skip to next
        check_exists = os.path.exists(filename)
        if check_exists is False:
            print('This path did not exists, skipping:{}'.format(filename))
            continue

        data.append(np.array(nb.load(filename).agg_data())[:num_channels,:])

        if np.isnan(data[i]).sum() != 0: # so if there are nans
            data_check_nans = np.isnan(data[i]).sum()
            nan_surf_subject_count += 1
            sub_ids_nan_vals.append(id)
            print(f"DATA SUBJ ID:{id}/ iter:{i} NaN Count - {data_check_nans}")

        if i%300==0:
            print('\nLoading GIFTI for subject: {}'.format(i))

elif chosen_hemi == '2LR':
    left_nan_list = []
    # left_nan_idx = []
    right_nan_list = []
    right_nan_idx = []

    clean_subj_ids = []
    print('Both hemispheres were chosen. Starting with Left.')
    for i,id in enumerate(ids):
        # data here becomes [iiL,iiR, ii+1L,ii+1R .... NL,NR] so shape is list of num_subs*2?
        if dataset == "ABCD":
            filename = os.path.join(f'{path_to_data}/resamp_{id}.L.shape.gii')
        elif dataset == "HCPYA_ABCDdr":
            filename = os.path.join(f'{path_to_data}/resamp_{id}.L.shape.gii')
        else:
            filename = os.path.join(path_to_data,'ICA_fs_LR_32',nm_fs_data,'resamp_{}.L.shape.gii'.format(id)) # shape.gii is a metric file for values at each vertex, in our case the ICA values
        
        # check if real, ow skip to next
        check_exists = os.path.exists(filename)
        if check_exists is False:
            print('This path did not exists, skipping:{}'.format(filename))
            continue
        
        curr_surf_data = np.array(nb.load(filename).agg_data())[:num_channels,:]
        if np.isnan(curr_surf_data).sum() != 0: # so if there are nans in currentt subj
            data_check_nans = np.isnan(curr_surf_data).sum()
            nan_surf_subject_count += 1
            # left_nan_list.append(id)
            # left_nan_idx.append(i)
            print(f"DATA SUBJ ID:{id}/ iter:{i} NaN Count - {data_check_nans}")
            continue # skip this and dont add to data list since NaNs anyway. Pre-emptive cleaning ig.

        data.append(curr_surf_data)
        
        clean_subj_ids.append(id) #subj id that is CLEAN and itll be LLLLL...RRRRR

        # print('Both hemispheres were chosen. Now doing with Right.')
    # l_order=0 #only load R hemis that had a clean L
    # for i,id in enumerate(ids): #ids > clean_subj_ids, so using that to idx with i - maybe not smartest cause assumes ids > clean_subj_ids to work
    #     # load rigth hemisphere
    #     if id != clean_subj_ids[l_order]: #check that for an id a left hemi has been saved (so clean L hemi)
    #         # print(f"Left hemi ID:{clean_subj_ids[l_order]}, idx:{l_order}")
    #         continue

        if dataset == "ABCD": #load the ID cause passed L hemi clean check
            filename = os.path.join(f'{path_to_data}/resamp_{id}.R.shape.gii')
        else:
            filename = os.path.join(path_to_data,'ICA_fs_LR_32',nm_fs_data,'resamp_{}.R.shape.gii'.format(id))

        curr_surf_data = np.array(nb.load(filename).agg_data())[:num_channels,:]
        # see if for some reason the L was fine but R has NaNs. 
        if np.isnan(curr_surf_data).sum() != 0: # so if there are nans in currentt subj
            data_check_nans = np.isnan(curr_surf_data).sum()
            nan_surf_subject_count += 1
            # right_nan_list.append(id)
            # right_nan_idx.append(i)
            print(f"DATA SUBJ ID:{id}/ iter:{i} NaN Count - {data_check_nans}")
            
            # in case somehow L is good but R has NaNs, delete L and skip this R to keep it clean for 2LR condition training. 
            del data[-1], clean_subj_ids[-1] # delete most recent which is L hemisphere for this subj
            # l_order += 1 
            continue
        
        data.append(curr_surf_data)
        clean_subj_ids.append(id) #subj id that is CLEAN and itll be LRLR..LR
        # l_order += 1 
        if i%300==0:
            print('\nLoading GIFTI for subject: {}'.format(i))

    print(f"After RIGHT data loading. clean_subj_ids: {len(clean_subj_ids)}")
    # print(f"After LEFT data loading. clean_subj_ids: {len(clean_subj_ids)}")

    assert len(data) == len(clean_subj_ids), "data and clean_subj_List don't match. Something wrong with NaN exclusion."

    ids = np.sort(np.concatenate((ids,ids))) # when doing 2LR, need to double IDS
    print(f"check sort ids to make sure irder is therefire LRLRLR..LR {ids[0:2]} {ids[-1:-3]}")
    print(f"ids shape: {ids.shape}")
    print(f"ids unique: {len(np.unique(ids))}")

    clean_subj_ids = np.asarray(clean_subj_ids) # no need to concat or sort because already LRLRLR..LR
    print(f"check sort clean_subj_ids to make sure irder is therefire LRLRLR...LR {clean_subj_ids}")
    print(f"clean_subj_ids shape: {clean_subj_ids.shape}")
    print(f"clean_subj_ids unique: {len(np.unique(clean_subj_ids))}")

    clean_ids = np.isin(ids, clean_subj_ids) #ids that ARE in clean_subj_ids
    print(f"check sort clean_ids to make sure order is therefore LRLR..LR {ids[0:2]} {ids[-1]}")
    print(f"clean_ids shape: {ids[clean_ids].shape}")
    print(f"clean_ids unique: {len(np.unique(ids[clean_ids]))}")
    df_version = pd.DataFrame(ids[clean_ids])
    df_version.to_csv(f'{sub_ids_path}/{split}_subj_IDs_clean_BOTH_{dataset}.csv')
    print(f"Clean subj count BOTH: - {ids[clean_ids].shape}.")

    data = np.asarray(data) # for 2LR data is already cleaned cause Subj with NaNs get skipped
    labels = labels_nan_clean[clean_ids] # boolean ind and it wokred on ids==shape==labels_nan_clean.
    print(f"2LR merged labels shape: {labels.shape}")
    print(f"2LR merged data shape: {data.shape}")

    ids = ids[clean_ids] #now make IDs bve the clean ones
    print(f"2LR merged IDs shape: {ids.shape}")

    num_subjects = len(np.unique(ids))
    print(f"Num of subjects is: {num_subjects}")
